# Trustee Exercise: Explaining ML Models in Cybersecurity

## Introduction

In this exercise, you'll learn how to use **Trustee** to explain machine learning models in cybersecurity applications.

### What is Trustee?

Trustee is a model explanation framework that extracts **interpretable decision trees** from complex black-box models (neural networks, random forests, gradient boosting, etc.).

### Why Explainability Matters in Security

- **Debugging:** Identify if your model learned the right patterns or shortcuts
- **Trust:** Understand why a network packet was classified as malicious
- **Compliance:** Meet regulatory requirements for explainable AI
- **Adversarial Robustness:** Detect vulnerabilities in your model's decision-making

### Exercise Structure

**Part 1: Demo** - Quick walkthrough of Trustee on a simple example

**Part 2: Your Turn** - Train and explain an OS fingerprinting model (network traffic classification)

---

# Part 1: Quick Demo - Understanding Trustee

Let's start with a simple example to understand how Trustee works.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn import tree
from trustee import ClassificationTrustee

# Set random seeds for reproducibility
np.random.seed(42)

print("✓ Libraries imported successfully!")

## Step 1: Create a Simple Dataset

In [ ]:
# Create a simple "moons" dataset
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Visualize the data
plt.figure(figsize=(8, 6))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], 
           c='blue', label='Class 0', alpha=0.6, edgecolors='k')
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], 
           c='red', label='Class 1', alpha=0.6, edgecolors='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Training Data: Two Moons')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## Step 2: Train a Random Forest (Black-Box Model)

In [ ]:
# Train a Random Forest with 100 trees
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Random Forest Test Accuracy: {accuracy:.3f}")
print(f"Model complexity: {rf_model.n_estimators} trees")

## Step 3: Use Trustee to Explain the Model

Now we'll extract a single, interpretable decision tree that mimics the Random Forest's behavior.

In [ ]:
# Initialize Trustee
print("Extracting decision tree explanation with Trustee...")
trustee = ClassificationTrustee(expert=rf_model)

# Fit Trustee on the training data
trustee.fit(
    X_train,
    y_train,
    num_iter=10,
    num_stability_iter=5,
    samples_size=0.3,
    verbose=True
)

# Extract explanation
dt, pruned_dt, agreement, reward = trustee.explain()

print(f"\n{'='*60}")
print(f"Trustee Explanation Metrics:")
print(f"{'='*60}")
print(f"Agreement (training fidelity): {agreement:.3f}")
print(f"Reward (validation fidelity): {reward:.3f}")
print(f"Pruned Decision Tree nodes: {pruned_dt.tree_.node_count}")
print(f"\nOriginal model: 100 trees → Explanation: 1 tree with {pruned_dt.tree_.node_count} nodes")

## Step 4: Visualize the Explanation

In [ ]:
# Visualize the decision tree
plt.figure(figsize=(15, 10))
tree.plot_tree(
    pruned_dt,
    feature_names=['Feature 1', 'Feature 2'],
    class_names=['Class 0', 'Class 1'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Trustee Decision Tree Explanation", fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Test the explanation
dt_pred = pruned_dt.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_pred)
fidelity = accuracy_score(y_pred, dt_pred)

print(f"Decision Tree Accuracy: {dt_accuracy:.3f}")
print(f"Fidelity to Random Forest: {fidelity:.3f}")
print(f"\n✓ The decision tree maintains high accuracy while being much simpler!")

### Key Takeaways from Demo

1. **Trustee extracts a simple decision tree** from a complex model (100 trees → 1 tree)
2. **High fidelity**: The explanation maintains the model's predictions
3. **Interpretable**: You can now understand exactly how the model makes decisions
4. **Same accuracy**: The decision tree performs nearly as well as the original model

---

---

# Part 2: Exercise - OS Fingerprinting

## Background

**OS Fingerprinting** identifies the operating system based on network traffic patterns. In this exercise:

- **Data:** nPrint features extracted from network packet captures (pcaps)
- **Features:** 416 network features (packet headers, TCP/IP fields, etc.)
- **Classes:** 4 operating systems (Linux, Windows, MacOS, Kali Linux)
- **Your Goal:** Train a classifier and use Trustee to understand which network features distinguish different operating systems

## Your Tasks

You will:
1. Load and explore the OS fingerprinting dataset
2. Train a Random Forest classifier
3. Apply Trustee to extract an explanation
4. Analyze which network features are most important
5. Answer analysis questions

---

## Task 1: Load and Explore the Dataset

In [1]:
import pickle

# Load the OS fingerprinting data
DATA_PATH = '../data/os_fingerprinting/'

print("Loading OS fingerprinting data...")

with open(f'{DATA_PATH}/X.pkl', 'rb') as f:
    X_os = pickle.load(f)
    
with open(f'{DATA_PATH}/y.pkl', 'rb') as f:
    y_os = pickle.load(f)

# Map numeric labels to OS names
os_label_map = {
    0: "Linux 3.11+",
    1: "Windows 7/8",
    2: "MacOS X",
    3: "Kali Linux"
}

print(f"✓ Data loaded successfully!")
print(f"  Samples: {X_os.shape[0]:,}")
print(f"  Features: {X_os.shape[1]}")
print(f"  Classes: {len(y_os.unique())}")
print(f"\nClass distribution:")
for label, count in y_os.value_counts().sort_index().items():
    print(f"  {os_label_map[label]}: {count:,} samples")

Loading OS fingerprinting data...
✓ Data loaded successfully!
  Samples: 7,836
  Features: 416
  Classes: 4

Class distribution:
  Linux 3.11+: 279 samples
  Windows 7/8: 3,787 samples
  MacOS X: 623 samples
  Kali Linux: 3,147 samples


In [ ]:
# Explore the features
print("First few feature names:")
print(X_os.columns[:10].tolist())
print(f"\n... and {X_os.shape[1] - 10} more features")

# Display sample data
print("\nSample data:")
X_os.head()

In [ ]:
# Split into train/test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_os, y_os, test_size=0.2, random_state=42, stratify=y_os
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

---

## Task 2: Train a Random Forest Classifier

**TODO:** Train a Random Forest model to classify operating systems based on network features.

**Hints:**
- Use `RandomForestClassifier` from sklearn
- Try `n_estimators=100` and `max_depth=10`
- Use `random_state=42` for reproducibility
- Consider using `n_jobs=-1` to use all CPU cores

In [ ]:
# TODO: Train a Random Forest classifier
# YOUR CODE HERE

print("Training Random Forest classifier...")

# Step 1: Create the RandomForestClassifier
os_model = None  # TODO: Initialize RandomForestClassifier with appropriate parameters

# Step 2: Train the model
# TODO: Fit the model on X_train and y_train

# Step 3: Make predictions on test set
# TODO: Get predictions for X_test
y_pred = None

# Step 4: Calculate and print accuracy
# TODO: Calculate accuracy_score
test_accuracy = None

print(f"\n✓ Model trained!")
print(f"Test Accuracy: {test_accuracy:.3f}")

# Print detailed classification report
print(f"\nClassification Report:")
print(classification_report(
    y_test, 
    y_pred,
    target_names=[os_label_map[i] for i in sorted(os_label_map.keys())]
))

---

## Task 3: Apply Trustee to Explain the Model

**TODO:** Use Trustee to extract a decision tree explanation from your Random Forest.

**Hints:**
- Initialize `ClassificationTrustee` with your trained model
- Use `num_iter=10`, `num_stability_iter=5`, `samples_size=0.3`
- Set `verbose=True` to see progress
- Call `.explain()` to get the decision tree

In [ ]:
# TODO: Apply Trustee to extract explanation
# YOUR CODE HERE

print("Applying Trustee to extract decision tree explanation...")

# Step 1: Initialize ClassificationTrustee
os_trustee = None  # TODO: Create ClassificationTrustee with your model

# Step 2: Fit Trustee on training data
# TODO: Call trustee.fit() with appropriate parameters

# Step 3: Extract explanation
# TODO: Call trustee.explain() to get dt, pruned_dt, agreement, reward
os_dt = None
os_pruned_dt = None
os_agreement = None
os_reward = None

print(f"\n{'='*60}")
print(f"Trustee Explanation Metrics:")
print(f"{'='*60}")
print(f"Agreement (training fidelity): {os_agreement:.3f}")
print(f"Reward (validation fidelity): {os_reward:.3f}")
print(f"Decision Tree nodes: {os_dt.tree_.node_count}")
print(f"Pruned Decision Tree nodes: {os_pruned_dt.tree_.node_count}")

---

## Task 4: Visualize and Analyze the Explanation

**TODO:** Visualize the decision tree and analyze feature importance.

In [ ]:
# Visualize the decision tree
fig, ax = plt.subplots(figsize=(25, 12))
tree.plot_tree(
    os_pruned_dt,
    class_names=[os_label_map[i] for i in sorted(os_label_map.keys())],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
plt.title("OS Fingerprinting Decision Tree Explanation", fontsize=18, pad=20)
plt.tight_layout()
plt.show()

print("\nDecision Tree Statistics:")
print(f"  Depth: {os_pruned_dt.get_depth()}")
print(f"  Leaves: {os_pruned_dt.get_n_leaves()}")
print(f"  Total nodes: {os_pruned_dt.tree_.node_count}")

In [ ]:
# TODO: Analyze feature importance
# YOUR CODE HERE

# Step 1: Get feature importance from the decision tree
os_feature_importance = None  # TODO: Get feature_importances_ from os_pruned_dt
os_feature_names = X_os.columns.tolist()

# Step 2: Rank features by importance
# TODO: Create a list of (feature_name, importance) tuples and sort by importance
os_top_features = None  # Get top 15 features

print("Top 15 Most Important Network Features for OS Fingerprinting:")
print("="*60)
for i, (feature, importance) in enumerate(os_top_features, 1):
    print(f"{i:2d}. {feature:40s}: {importance:.4f}")

In [ ]:
# Visualize top features and performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 feature importance bar chart
features_top10, importances_top10 = zip(*os_top_features[:10])
axes[0].barh(range(len(features_top10)), importances_top10, color='steelblue')
axes[0].set_yticks(range(len(features_top10)))
axes[0].set_yticklabels(features_top10, fontsize=9)
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('Top 10 Network Features for OS Classification', fontsize=14)
axes[0].invert_yaxis()

# Performance comparison
os_dt_pred = os_pruned_dt.predict(X_test)
os_dt_accuracy = accuracy_score(y_test, os_dt_pred)
os_fidelity = accuracy_score(y_pred, os_dt_pred)

metrics = ['Original RF\nAccuracy', 'Decision Tree\nAccuracy', 'Fidelity\n(Agreement)']
values = [test_accuracy, os_dt_accuracy, os_fidelity]
colors = ['#2ecc71', '#3498db', '#e74c3c']

axes[1].bar(metrics, values, color=colors, alpha=0.7)
axes[1].set_ylim([0, 1.0])
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Model Performance Comparison', fontsize=14)
axes[1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
for i, v in enumerate(values):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("Key Metrics:")
print(f"{'='*60}")
print(f"✓ Decision tree achieves {os_fidelity:.1%} fidelity to Random Forest")
print(f"✓ Simplified to {os_pruned_dt.tree_.node_count} nodes (vs 100 trees in RF)")
print(f"✓ Top features reveal OS-specific network stack behaviors")

---

## Summary


✓ Use Trustee to extract interpretable explanations from complex models

✓ Train and evaluate a machine learning model for OS fingerprinting

✓ Analyze feature importance to understand model decisions

✓ Identify potential security vulnerabilities through explainability

### Key Takeaways

1. **Explainability is crucial for security:** You need to understand *why* a model makes predictions
2. **Trustee provides high-fidelity explanations:** Decision trees can approximate complex models accurately
3. **Feature importance reveals patterns:** Understanding which features matter helps identify vulnerabilities
4. **Simpler is better:** A single decision tree is much easier to audit than 100 trees

### Further Reading

- [Trustee Documentation](https://trusteeml.github.io)
- [Trustee GitHub](https://github.com/TrusteeML/trustee)
- [Emperor Repository](https://github.com/TrusteeML/emperor) - More examples
- Paper: "Trustee: A Framework for Model Interpretation" (FAT* 2022)